# Notebook 03 — Long answer review

You write a paragraph-length answer to a question. The model reads it
alongside retrieved course context, then gives you a score (1–5) and
specific feedback about what you got right and wrong.

**Honest note on small models:**
flan-t5-small/base often gives generic or nonsensical feedback for this task.
TinyLlama and larger models do much better. For reliable scoring, the notebook
also includes a `gpt4_review` option — this is the cleanest fallback and what
you'd use in production.

**Try this:** Write intentionally incomplete answers and see if the model
catches the missing parts.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

MODEL_CHOICE = "cpu_tiny"      # WARNING: small models give weak feedback
# MODEL_CHOICE = "small_hf"
# MODEL_CHOICE = "larger_hf"  # Recommended minimum for this task
# MODEL_CHOICE = "own_model"
# MODEL_CHOICE = "gpt4"       # Uses OpenAI API — most reliable

CHECKPOINT_PATH  = "../teaching_assistant/checkpoints/step_005000.pt"
RAG_INDEX_DIR    = "../teaching_assistant/rag_index"
OPENAI_API_KEY   = "sk-..."   # only needed if MODEL_CHOICE == "gpt4"

# The question and your answer to review
QUESTION = "What is dropout and why is it used in neural networks?"

STUDENT_ANSWER = """
Dropout is when you turn off some neurons randomly during training.
It helps prevent the model from memorizing the training data.
"""
# ^ Intentionally incomplete: doesn't mention co-adaptation, inference behaviour,
#   the keep probability, or that it acts as an ensemble. Good test for the reviewer.

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL LOADER — compatible with transformers >= 4.40
#
# text2text-generation was removed from the pipeline registry
# in newer transformers. We use the model + tokenizer directly
# instead — it's the same thing under the hood, just explicit.
# ═══════════════════════════════════════════════════════════
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if MODEL_CHOICE == "cpu_tiny":
    # ── flan-t5-small ──────────────────────────────────────
    # T5 is an encoder-decoder (seq2seq) model.
    # We encode the prompt with the encoder, then let the
    # decoder generate the output autoregressively.
    # ~80 MB, runs in seconds on CPU.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
    _mdl.eval()
    # Note: T5 stays on CPU (device=-1 was the old pipeline arg)

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "small_hf":
    # ── flan-t5-base ───────────────────────────────────────
    # Same architecture, 3× more parameters. Better at following
    # structured instructions (e.g. "output only JSON").
    # ~250 MB. CPU works but is slow; GPU recommended.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
    _mdl = _mdl.to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "larger_hf":
    # ── TinyLlama-1.1B-Chat ────────────────────────────────
    # Decoder-only (same family as GPT). Uses a chat template
    # so the prompt is wrapped in <|user|> / <|assistant|> tags.
    # ~2.2 GB VRAM — fits easily on a 3080 laptop.
    # Swap model_id for "microsoft/phi-2" (2.7B) for better quality.
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    _tok = AutoTokenizer.from_pretrained(model_id)
    _mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        # Apply the chat template so TinyLlama knows this is a user message
        messages  = [{"role": "user", "content": prompt}]
        formatted = _tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tok(formatted, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
            )
        # Decode only the newly generated tokens (skip the prompt)
        new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
        return _tok.decode(new_ids, skip_special_tokens=True)

elif MODEL_CHOICE == "own_model":
    # ── Your trained model from train.py ───────────────────
    # Uses tiktoken gpt2 BPE tokenizer (same as training).
    import sys
    sys.path.insert(0, "../teaching_assistant")
    from rag_pipeline import load_model
    import tiktoken
    _own_model, _own_tok, _own_cfg = load_model(CHECKPOINT_PATH, device)
    _enc = tiktoken.get_encoding("gpt2")

    def generate(prompt, max_new_tokens=200):
        ids = _enc.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)
        out = _own_model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=50,
            stop_token=_enc.eot_token,
        )
        new_ids = out[0, len(ids):].tolist()
        return _enc.decode(new_ids).replace("<|endoftext|>", "").strip()

print(f"Model ready: {MODEL_CHOICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD RAG
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "../teaching_assistant")
retriever = None
try:
    from retriever import Retriever
    retriever = Retriever(index_dir=RAG_INDEX_DIR)
    print("RAG loaded")
except Exception as e:
    print(f"RAG unavailable ({e})")

In [ ]:
# ═══════════════════════════════════════════════════════════
# REVIEW FUNCTION
# The prompt says explicitly: score 1-5, then explain what
# was right and what was missing. Low temperature keeps scoring
# consistent across runs.
# ═══════════════════════════════════════════════════════════
import re

REVIEW_TEMPLATE = """You are a machine learning teacher reviewing a student's answer.

{context_section}Question: {question}

Student's answer:
{student_answer}

Review the answer. Give:
1. A score from 1 (completely wrong) to 5 (excellent)
2. One specific thing the student got right
3. One important thing that is missing or incorrect

Start your response with: Score: X/5"""


def review_answer(question, student_answer, use_rag=True):
    context_section = ""
    if use_rag and retriever is not None:
        chunks = retriever.query(question, top_k=2)
        context = "\n".join(c["text"][:300] for c in chunks)
        context_section = f"Reference material:\n{context}\n\n"

    prompt = REVIEW_TEMPLATE.format(
        context_section=context_section,
        question=question,
        student_answer=student_answer.strip(),
    )

    if MODEL_CHOICE == "gpt4":
        import json
        resp = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3, max_tokens=250,
        )
        raw = resp.choices[0].message.content
    else:
        raw = generate(prompt)

    # Parse the score if present
    score = None
    m = re.search(r'[Ss]core[:\s]+(\d)[/\s]*5', raw)
    if m:
        score = int(m.group(1))

    return {"raw": raw, "score": score}


# ── Run it ─────────────────────────────────────────────────
print(f"Question: {QUESTION}")
print(f"\nStudent answer: {STUDENT_ANSWER.strip()}")
print("\n" + "─"*50)

result = review_answer(QUESTION, STUDENT_ANSWER, use_rag=True)
print(f"Score: {result['score']}/5" if result['score'] else "(score not parsed)")
print("\nFull review:")
print(result["raw"])

In [ ]:
# ═══════════════════════════════════════════════════════════
# TEST DIFFERENT QUALITY ANSWERS
# See how the scores change for a bad, partial, and good answer.
# ═══════════════════════════════════════════════════════════

test_answers = [
    # (label, answer)
    ("Bad", "Dropout removes neurons from the network permanently."),
    ("Partial", "Dropout randomly disables neurons during training to prevent overfitting."),
    ("Good",
     """Dropout is a regularization technique where a random fraction p of neuron
     activations are set to zero during each training step. This prevents neurons
     from co-adapting — each neuron must learn to be useful independently. At
     inference, dropout is disabled and weights are scaled by (1-p). Dropout can
     be interpreted as an ensemble of exponentially many sub-networks."""),
]

q = "What is dropout and why is it used in neural networks?"

for label, ans in test_answers:
    r = review_answer(q, ans, use_rag=True)
    score_str = f"{r['score']}/5" if r['score'] else "?"
    print(f"[{label}] Score: {score_str}")
    print(f"  {r['raw'][:200].strip()}\n")